In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np

In [ ]:
raw_datasets = load_dataset("Intel/polite-guard")
train_valid_test_dataset = DatasetDict({
    'train': raw_datasets['train'],
    'validation': raw_datasets['validation'],
    'test': raw_datasets['test']
})

label_list = raw_datasets["train"].unique("label")
label_list.sort()
label2id = {lbl: i for i, lbl in enumerate(label_list)}

In [ ]:
def my_preprocess_function(tokenizer):
    def apply(sample):
        toks = tokenizer(sample["text"], truncation=True, padding=True)
        labels = [label2id[l] for l in sample["label"]]
        toks["labels"] = labels
        return toks
    return apply

In [ ]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_dataset = train_valid_test_dataset.map(
    my_preprocess_function(tokenizer),
    batched=True,
    remove_columns=["text", "source", "reasoning", "label"],
)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

In [ ]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="weighted")

training_args = TrainingArguments(
    output_dir="./polite_guard_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.predict(test_dataset=tokenized_dataset["test"])